# TripAdvisor Information Retrieval Project

Alvaro SERERO, Leo WINTER

ESILV A4 DIA6

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
DATA_PATH = Path("TripAdvisorTrainingDataProject1")
RANDOM_STATE = 42

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## Load Data

In [38]:
attraction_sub_categorie = pd.read_csv(DATA_PATH / "AttractionSubCategorie.csv")
attraction_sub_type = pd.read_csv(DATA_PATH / "AttractionSubType.csv")
cuisine_df = pd.read_csv(DATA_PATH / "cuisine.csv")
dietary_restrictions_df = pd.read_csv(DATA_PATH / "dietary_restrictions.csv")
restaurant_type_df = pd.read_csv(DATA_PATH / "restaurantType.csv")
reviews = pd.read_csv(DATA_PATH / "reviews83325.csv", low_memory=False) 
trip_advisor = pd.read_csv(DATA_PATH / "Tripadvisor.csv", low_memory=False) 

print(f"reviews: {reviews.shape}")
print(f"trip_advisor: {trip_advisor.shape}")
print("\ntrip_advisor columns:", trip_advisor.columns.tolist())

reviews: (340385, 21)
trip_advisor: (3761, 60)

trip_advisor columns: ['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis', 'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse', 'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars', 'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete', 'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance', 'hotelbearing', 'restaurantTypeCuisine', 'restaurantDietaryRestrictions', 'restaurantMeals', 'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService', 'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType', 'activiteSubCategorie', 'activiteSubType', 'website', 'nbScanReview', 'dateLastScanReviews', 'shape_gid', 'gadm36_gid', 'hotelprice', 'hotelBookingID', 'restaurantSubcategory', 'restaurantType', 'ap_additional_info', 'ap_age_band_list', 'ap_attraction_ids', 'ap_booking_question_list', 'ap_bubble_rating_integer', 'ap_duration', 'ap_exclusion', 'ap_inclusions', 'ap_introduction',

## Exploratory Data Analysis

In [40]:
print("Place type distribution (typeR):")
print(trip_advisor["typeR"].value_counts())
# H = Hotel | R = Restaurant | A = Attraction | AP = Attraction Product

Place type distribution (typeR):
typeR
AP    2669
R      623
A      405
H       64
Name: count, dtype: int64


In [41]:
print("\nReview language distribution (top 10):")
print(reviews["langue"].value_counts(normalize=True).mul(100).round(2).head(10))


Review language distribution (top 10):
langue
en    44.97
fr    29.11
it     6.73
es     6.40
pt     5.73
ru     1.60
de     1.54
ja     1.21
nl     0.83
ko     0.33
Name: proportion, dtype: float64


In [43]:
def add_name_from_id(dataset_input, input_column_name, dataset_id):
    """
    Map a column of comma-separated IDs to human-readable names.
    Drops the original ID column; adds a new <col>_name column.
    Always called on a copy — never mutates the source DataFrame.
    """
    if input_column_name not in dataset_input.columns:
        print(f"Column '{input_column_name}' not found — skipping.")
        return dataset_input

    dataset_id = dataset_id.copy()
    dataset_id["id"] = dataset_id["id"].astype(str)
    mapping_id = dataset_id.set_index("id")["name"].to_dict()

    def map_ids(id_string):
        if pd.isna(id_string):
            return np.nan
        parts = str(id_string).split(",")
        return ", ".join(mapping_id.get(p.strip(), "Unknown") for p in parts)

    out = dataset_input.copy()
    out[f"{input_column_name}_name"] = out[input_column_name].apply(map_ids)
    return out.drop(columns=[input_column_name])


# Build a named copy for display only — trip_advisor stays untouched
trip_advisor_named = trip_advisor.copy()
trip_advisor_named = add_name_from_id(trip_advisor_named, "activiteSubCategorie", attraction_sub_categorie)
trip_advisor_named = add_name_from_id(trip_advisor_named, "activiteSubType", attraction_sub_type)
trip_advisor_named = add_name_from_id(trip_advisor_named, "restaurantTypeCuisine", cuisine_df)
trip_advisor_named = add_name_from_id(trip_advisor_named, "restaurantDietaryRestrictions", dietary_restrictions_df)
trip_advisor_named = add_name_from_id(trip_advisor_named, "restaurantType", restaurant_type_df)

print("Sample (human-readable categories):")
print(trip_advisor_named[["id", "nom", "typeR",
                           "activiteSubCategorie_name", "restaurantType_name"]].head(5))

Sample (human-readable categories):
       id                       nom typeR  \
0  188467          Place des Vosges     A   
1  188468  Rue des Francs Bourgeois     A   
2  188470        Village Saint-Paul     A   
3  188471          Au Passe-partout     A   
4  188472     Cloître des Billettes     A   

             activiteSubCategorie_name restaurantType_name  
0                   Sites touristiques                 NaN  
1                   Sites touristiques                 NaN  
2  Shopping, Sites touristiques, Autre                 NaN  
3                             Shopping                 NaN  
4                   Sites touristiques                 NaN  


### Creating trip_advisor dataset (test dataset)

In [12]:
print("ActiviteSubCategorie :")
print("     Nan values",attraction_sub_categorie["name"].isna().sum())
print("     Unique values",attraction_sub_categorie["name"].nunique())
print("     Total values",attraction_sub_categorie["name"].count())
print("     id type :", attraction_sub_categorie['id'].dtype)

print("\nActiviteSubType :")
print("     Nan values",attraction_sub_type["name"].isna().sum())
print("     Unique values",attraction_sub_type["name"].nunique())
print("     Total values",attraction_sub_type["name"].count())
print("     id type :", attraction_sub_type['id'].dtype)

print("\nRestaurantTypeCuisine :")
print("     Nan values",cuisine["name"].isna().sum())
print("     Unique values",cuisine["name"].nunique())
print("     Total values",cuisine["name"].count())
print("     id type :", cuisine['id'].dtype)

print("\nRestaurantDietaryRestrictions :")
print("     Nan values",dietary_restrictions["name"].isna().sum())
print("     Unique values",dietary_restrictions["name"].nunique())
print("     Total values",dietary_restrictions["name"].count())
print("     id type :", dietary_restrictions['id'].dtype)

print("\nRestaurantType :")
print("     Nan values",restaurant_type["name"].isna().sum())
print("     Unique values",restaurant_type["name"].nunique())
print("     Total values",restaurant_type["name"].count())
print("     id type :", restaurant_type['id'].dtype)


ActiviteSubCategorie :
     Nan values 0
     Unique values 20
     Total values 20
     id type : int64

ActiviteSubType :
     Nan values 0
     Unique values 229
     Total values 229
     id type : int64

RestaurantTypeCuisine :
     Nan values 0
     Unique values 161
     Total values 161
     id type : int64

RestaurantDietaryRestrictions :
     Nan values 0
     Unique values 5
     Total values 5
     id type : int64

RestaurantType :
     Nan values 0
     Unique values 9
     Total values 9
     id type : int64


In [13]:
print("ActiviteSubCategorie :")
print("     Unique values :",trip_advisor['activiteSubCategorie'].nunique())
print("     Nan values",trip_advisor['activiteSubCategorie'].isna().sum())
print("     Total values :",trip_advisor['activiteSubCategorie'].count())
print("     id type :", trip_advisor['activiteSubCategorie'].dtype)

print("\nActiviteSubType :")
print("     Unique values :",trip_advisor['activiteSubType'].nunique())
print("     Nan values",trip_advisor['activiteSubType'].isna().sum())
print("     Total values :",trip_advisor['activiteSubType'].count())
print("     id type :", trip_advisor['activiteSubType'].dtype)

print("\nRestaurantTypeCuisine :")
print("     Unique values :",trip_advisor['restaurantTypeCuisine'].nunique())
print("     Nan values",trip_advisor['restaurantTypeCuisine'].isna().sum())
print("     Total values :",trip_advisor['restaurantTypeCuisine'].count())
print("     id type :", trip_advisor['restaurantTypeCuisine'].dtype)

print("\nRestaurantDietaryRestrictions :")
print("     Unique values :",trip_advisor['restaurantDietaryRestrictions'].nunique())
print("     Nan values",trip_advisor['restaurantDietaryRestrictions'].isna().sum())
print("     Total values :",trip_advisor['restaurantDietaryRestrictions'].count())
print("     id type :", trip_advisor['restaurantDietaryRestrictions'].dtype)

print("\nRestaurantType :")
print("     Unique values :",trip_advisor['restaurantType'].nunique())
print("     Nan values",trip_advisor['restaurantType'].isna().sum())
print("     Total values :",trip_advisor['restaurantType'].count())
print("     id type :", trip_advisor['restaurantType'].dtype)

ActiviteSubCategorie :
     Unique values : 45
     Nan values 3356
     Total values : 405
     id type : object

ActiviteSubType :
     Unique values : 124
     Nan values 3356
     Total values : 405
     id type : object

RestaurantTypeCuisine :
     Unique values : 245
     Nan values 3243
     Total values : 518
     id type : object

RestaurantDietaryRestrictions :
     Unique values : 16
     Nan values 3529
     Total values : 232
     id type : object

RestaurantType :
     Unique values : 14
     Nan values 3138
     Total values : 623
     id type : object


## Preprocessing
- English reviews only
- drop rows with null reviews
- lowercase
- strip punctuation
- remove stopwords

In [22]:
# Keep only english reviews
reviews_en = reviews[reviews['langue'] == 'en'].copy()
print(f'English reviews: {len(reviews_en)} / {len(reviews)} total')

English reviews: 153071 / 340385 total


In [28]:
TEXT_COL = 'review'

# Drop rows with null review text
reviews_en = reviews_en.dropna(subset=[TEXT_COL])
print(f'After dropping nulls: {len(reviews_en)}')

After dropping nulls: 153071


In [36]:
# Lowercase, remove punctuation/numbers, remove stopwords, optional stemming.

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text: str) -> str:
    """Lowercase, remove punctuation/numbers, remove stopwords, optional stemming."""

    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # we keep only letters
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    # Stemming
    # tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

reviews_en['clean_text'] = reviews_en[TEXT_COL].apply(clean_text)
print("Sample cleaned review:",reviews_en['clean_text'].iloc[0])

Sample cleaned review: personally think beautiful square paris well maintained area around gives opportunities grab bite eat well


## Build Place Documents

Each document contains all cleaned reviews of a place concatenated.

We also apply a cap to reduce the size imbalance between popular and less popular places.

In [37]:
# Column linking reviews to places
ID_COL_REVIEWS = 'idplace'
ID_COL_PLACES  = 'id'

# Optional: cap number of reviews per place to limit imbalance
MAX_REVIEWS_PER_PLACE = 50   # set to None to use all

def aggregate_reviews(group, max_reviews=MAX_REVIEWS_PER_PLACE):
    texts = group['clean_text'].tolist()
    if max_reviews:
        texts = texts[:max_reviews]
    return ' '.join(texts)

print('Aggregating reviews by place...')
place_docs = (
    reviews_en
    .groupby(ID_COL_REVIEWS)
    .apply(aggregate_reviews)
    .reset_index()
    .rename(columns={ID_COL_REVIEWS: 'place_id', 0: 'document'})
)
print(f'Places with at least 1 English review: {len(place_docs)}')

# Merge with places metadata (for evaluation only)
place_docs = place_docs.merge(
    trip_advisor[[ID_COL_PLACES, 'typeR', 'activiteSubCategorie', 'activiteSubType',
            'restaurantType', 'cuisine', 'priceRange']].rename(columns={ID_COL_PLACES: 'place_id'}),
    on='place_id', how='left'
)
print(f'After metadata merge: {len(place_docs)}')
print(place_docs['typeR'].value_counts())

Aggregating reviews by place...
Places with at least 1 English review: 1835


/var/folders/cr/3y3t2vr97xsf_f_y_xhw2jv00000gn/T/ipykernel_31134/2626100822.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregate_reviews)


KeyError: "['cuisine'] not in index"

In [14]:
def add_name_from_id(dataset_input, input_column_name, dataset_id):
    if input_column_name not in dataset_input:
        print(f"No {input_column_name}")
        return dataset_input
    
    def mapping_multiple(index_string):
        if pd.isna(index_string):
            return np.nan
        
        ids_list = str(index_string).split(',')
        names = [mapping_id.get(i.strip(), "Unknown") for i in ids_list]
        
        return ", ".join(names)
    
    dataset_id['id'] = dataset_id['id'].astype(str)
    mapping_id = dataset_id.set_index('id')['name'].to_dict()

    dataset_input[f"{input_column_name}_name"] = dataset_input[input_column_name].apply(mapping_multiple)
    
    return dataset_input.drop(columns=[input_column_name]) 

In [15]:
trip_advisor = add_name_from_id(trip_advisor,"activiteSubCategorie",attraction_sub_categorie)
trip_advisor = add_name_from_id(trip_advisor,"activiteSubType",attraction_sub_type) 
trip_advisor = add_name_from_id(trip_advisor,"restaurantTypeCuisine",cuisine)       
trip_advisor = add_name_from_id(trip_advisor,"restaurantDietaryRestrictions",dietary_restrictions) 
trip_advisor = add_name_from_id(trip_advisor,"restaurantType",restaurant_type) 

In [16]:
print("ActiviteSubCategorie :")
print("     Number of unique values :",trip_advisor['activiteSubCategorie_name'].nunique())
print("     Total not Nan values :",trip_advisor['activiteSubCategorie_name'].count())
print("     Nan values",trip_advisor['activiteSubCategorie_name'].isna().sum())
print("     id type :", trip_advisor['activiteSubCategorie_name'].dtype)
#print("     Unique values :",trip_advisor['activiteSubCategorie_name'].unique())

print("\nActiviteSubType :")
print("     Number of unique values :",trip_advisor['activiteSubType_name'].nunique())
print("     Total not Nan values :",trip_advisor['activiteSubType_name'].count())
print("     Nan values",trip_advisor['activiteSubType_name'].isna().sum())
print("     id type :", trip_advisor['activiteSubType_name'].dtype)
#print("     Unique values :",trip_advisor['activiteSubType_name'].unique())

print("\nRestaurantTypeCuisine :")
print("     Number of unique values :",trip_advisor['restaurantTypeCuisine_name'].nunique())
print("     Total not Nan values :",trip_advisor['restaurantTypeCuisine_name'].count())
print("     Nan values",trip_advisor['restaurantTypeCuisine_name'].isna().sum())
print("     id type :", trip_advisor['restaurantTypeCuisine_name'].dtype)
#print("     Unique values :",trip_advisor['restaurantTypeCuisine_name'].unique())

print("\nRestaurantDietaryRestrictions :")
print("     Number of unique values :",trip_advisor['restaurantDietaryRestrictions_name'].nunique())
print("     Total not Nan values :",trip_advisor['restaurantDietaryRestrictions_name'].count())
print("     Nan values",trip_advisor['restaurantDietaryRestrictions_name'].isna().sum())
print("     id type :", trip_advisor['restaurantDietaryRestrictions_name'].dtype)
#print("     Unique values :",trip_advisor['restaurantDietaryRestrictions_name'].unique())

print("\nRestaurantType :")
print("     Number of unique values :",trip_advisor['restaurantType_name'].nunique())
print("     Total not Nan values :",trip_advisor['restaurantType_name'].count())
print("     Nan values",trip_advisor['restaurantType_name'].isna().sum())
print("     id type :", trip_advisor['restaurantType_name'].dtype)
#print("     Unique values :",trip_advisor['restaurantType_name'].unique())

ActiviteSubCategorie :
     Number of unique values : 45
     Total not Nan values : 405
     Nan values 3356
     id type : object

ActiviteSubType :
     Number of unique values : 124
     Total not Nan values : 405
     Nan values 3356
     id type : object

RestaurantTypeCuisine :
     Number of unique values : 245
     Total not Nan values : 518
     Nan values 3243
     id type : object

RestaurantDietaryRestrictions :
     Number of unique values : 16
     Total not Nan values : 232
     Nan values 3529
     id type : object

RestaurantType :
     Number of unique values : 14
     Total not Nan values : 623
     Nan values 3138
     id type : object


In [17]:
non_nan_count = trip_advisor['restaurantType_name'].notna().sum()

### Creating the review and the reviews by place dataset

In [18]:
print("Number of languages in reviews:", reviews["langue"].nunique())
print("Languages in reviews:\n", reviews['langue'].value_counts(normalize=True)*100)

Number of languages in reviews: 28
Languages in reviews:
 langue
en      44.969960
fr      29.107628
it       6.731201
es       6.395111
pt       5.734977
ru       1.604947
de       1.538552
ja       1.211569
nl       0.832880
ko       0.332858
sv       0.270282
da       0.195661
el       0.167457
tr       0.167457
pl       0.143955
zhTW     0.143367
no       0.125152
zhCN     0.119864
iw       0.054350
cs       0.023797
th       0.023797
ar       0.022915
fi       0.022328
hu       0.021446
in       0.017921
sr       0.009107
sk       0.008520
vi       0.002938
Name: proportion, dtype: float64


In [19]:
LABEL_COLUMNS = ["typeR"]
def get_label(reviews :pd.DataFrame, airbnb_info: pd.DataFrame) -> pd.DataFrame:
    label_columns = [col for col in LABEL_COLUMNS if col in airbnb_info.columns]
    
    if not label_columns:
        print("Error: None of the label columns are in airbnb_info")
        return reviews

    temp_airbnb_info = airbnb_info[['id'] + label_columns].copy()

    temp_airbnb_info['label'] = temp_airbnb_info[LABEL_COLUMNS].apply(
    lambda row: [str(val) if pd.notna(val) and str(val).lower() != 'nan' else "unknown" for val in row], 
    axis=1)

    reviews = reviews.merge(temp_airbnb_info[['id', 'label']],
    left_on='idplace', 
    right_on='id', 
    how='left')

    reviews = reviews.drop("id", axis=1)
    return reviews

In [20]:
REVIEWS_COLUMNS = ["idplace","review","langue"]
def preprocess_reviews(reviews,trip_advisor):

    reviews = reviews[[REVIEWS_COLUMNS]] 
    reviews_by_place = reviews.groupby('idplace').agg({'review': list,'langue': list}).reset_index()
    reviews_by_place = get_label(reviews_by_place,trip_advisor)
    return reviews

In [21]:
tab_reviews = preprocess_reviews(reviews,trip_advisor)

KeyError: "None of [Index([('idplace', 'review', 'langue')], dtype='object')] are in the [columns]"

### Model

In [ ]:
from spacy import load
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import gc

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
PRINCIPAL_LANGUAGE = ["en", "fr"] # 74,17%
SECONDARY_LANGUAGE = ["it", "es", "pt"] # 18,85%
TERTIARY_LANGUAGE = ["ru", "de", "ja", "nl"] # 5,17%
OTHER_LANGUAGE_BIG = ["ko", "sv", "da", "el", "tr", "pl", "zhTW", "no", "zhCN"]   # 2,51%
OTHER_LANGUAGE_SMALL = ["hu", "iw", "fi", "cs", "ar", "th", "in", "sr", "vi", "sk"] # 0,24%

nlp_models = {
    "en": load("en_core_web_sm"),
    "fr": load("fr_core_news_sm"),
    "it": load("it_core_news_sm"),
    "es": load("es_core_news_sm"),
    "pt": load("pt_core_news_sm")
}

In [ ]:
def create_test_train(reviews: pd.DataFrame, percent:int, random_state:int = None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    test_size = percent / 100

    X_train,X_test = train_test_split(
        reviews, 
        test_size=test_size, 
        random_state=random_state,
        shuffle=True
    )

    y_train = X_train["label"]
    y_test = X_test["label"]
    X_train = X_train.drop("label",axis=1)
    X_test = X_test.drop("label",axis=1)
 
    return X_train,X_test,y_train,y_test

PRINCIPAL_LANGUAGES = ["en", "fr"] -> 74,17%   
SECONDARY_LANGUAGES = ["it", "es", "pt"] -> 18,85%   
TERTIARY_LANGUAGES = ["ru", "de", "ja", "nl"] -> 5,17%   
OTHER_LANGUAGES_BIG = ["ko", "sv", "da", "el", "tr", "pl", "zhTW", "no", "zhCN"]   -> 2,51%   
OTHER_LANGUAGES_SMALL = ["hu", "iw", "fi", "cs", "ar", "th", "in", "sr", "vi", "sk"] -> 0,24%   

In [ ]:
def sentences_to_world(reviews: list):
    world_sentences = []
    for review in reviews:
        nlp = nlp_models.get("en", nlp_models["en"])
        
        doc = nlp(review.lower())
        world_lemmas = [t.lemma_ for t in doc if t.is_alpha and not t.is_stop]
        world_sentences.append(" ".join(world_lemmas))
        
    return world_sentences


def model_nlp_advanced(reviews: pd.DataFrame, k: int = 5, max_features: int=1000):
    # A mettre dans une classe
    final_text = []
    for _, row in reviews.iterrows():
        cleaned_list = sentences_to_world(row['review'])
        final_text.append(" ".join(cleaned_list))
    
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(final_text)
    gc.collect()
    
    return tfidf_matrix, vectorizer
    
def get_similarities(matrix, k_neighbors: int, batch_size: int = 500):
    n_rows = matrix.shape[0]
    all_topk = []

    for i in range(0, n_rows, batch_size):
        batch = matrix[i : i + batch_size]

        sims_batch = (batch @ matrix.T).toarray()

        for j in range(sims_batch.shape[0]):
            global_idx = i + j
            if global_idx < sims_batch.shape[1]:
                sims_batch[j, global_idx] = -1 

        k_eff = min(k_neighbors, sims_batch.shape[1] - 1)
        part = np.argpartition(sims_batch, -k_eff, axis=1)[:, -k_eff:]
        
        row_indices = np.arange(sims_batch.shape[0])[:, None]
        topk_in_part = np.argsort(sims_batch[row_indices, part], axis=1)[:, ::-1]
        
        all_topk.append(part[row_indices, topk_in_part])

    return np.vstack(all_topk)